# M07: PySpark for Data Engineering

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sunilmogadati/systems-in-production/blob/main/implementation/notebooks/PySpark.ipynb)
[![View on GitHub](https://img.shields.io/badge/GitHub-View_on_GitHub-blue?logo=github)](https://github.com/sunilmogadati/systems-in-production/blob/main/implementation/notebooks/PySpark.ipynb)

**Program:** Data Engineering Reference
**Author:** Sunil Mogadati
**Module:** 07 of 13 (M00–M13)
**Prerequisites:** M03 (Python for DE), M04 (Advanced SQL), M05 (Data Modeling)
**Estimated time:** 6–8 hours

> **PySpark is the core processing engine for modern data engineering.** This module combines fundamentals and optimization — you learn to write PySpark AND make it performant.
Playbook: [PySpark](../../playbooks/data/pyspark/README.md)


---
## What This Module Covers

The PySpark content lives in the **M03: Python for Data Engineering** notebook (Sections 18–22, 25). This is by design — PySpark is built on Python, and learning them in the same notebook shows the progression from Python → pandas → PySpark.

| Section | Topic | What You'll Learn |
|---|---|---|
| **18** | PySpark Fundamentals | Spark architecture, DataFrames, transformations vs actions, lazy evaluation |
| **19** | PySpark Joins, Aggregations & Windows | Joins, GROUP BY, window functions (same concepts as M04 SQL, now in PySpark) |
| **20** | PySpark I/O & Delta Lake Preview | Read/write CSV, JSON, Parquet. Partition strategy. Delta Lake intro. |
| **21** | PySpark SQL | SQL interface on DataFrames — write SQL against Spark tables |
| **22** | PySpark Optimization | Execution plans, broadcast joins, caching, data skew |
| **25** | ETL Pipeline Patterns | Build ETL scripts, testing patterns, error handling |

**Open the notebook:**

[![Open M03 in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sunilmogadati/systems-in-production/blob/main/implementation/notebooks/Python_NumPy_Pandas.ipynb)

Scroll to **Section 18: PySpark Fundamentals** to begin.

---
## Why PySpark? (Before You Start)

> **In plain English:** You learned SQL (M04) — that's how you query data. You learned Python (M03) — that's how you script and automate. PySpark is what happens when your data is too big for SQL on a single database or too big for pandas on your laptop. PySpark spreads the work across many machines.

| Tool | Data Size | Runs On | When to Use |
|---|---|---|---|
| **pandas** | < 1 GB | Your laptop (single machine) | Quick analysis, prototyping |
| **SQL** | < 100 GB | Database server (single machine) | Structured queries, reporting |
| **PySpark** | 1 GB – petabytes | Cluster (many machines) | ETL pipelines, large-scale transforms |

**In M08 (next module), you'll run PySpark on:**
- **Dataproc** (GCP) — managed Spark cluster
- **EMR** (AWS) — managed Spark cluster

The PySpark code you write here runs identically on your laptop, on Dataproc, and on EMR. That's the power of Spark — same code, different scale.

---
## The SQL Developer’s Bridge

Section 12 of the M03 notebook contains a **4-column comparison table**: SQL → Python → pandas → PySpark.

If you know SQL, you already know PySpark concepts:

| SQL | PySpark |
|---|---|
| `SELECT col1, col2` | `df.select("col1", "col2")` |
| `WHERE duration > 60` | `df.filter(col("duration") > 60)` |
| `GROUP BY dnis` | `df.groupBy("dnis")` |
| `ORDER BY total DESC` | `df.orderBy(col("total").desc())` |
| `JOIN orders ON call_id` | `df.join(orders, "call_id")` |
| `COUNT(*), AVG(duration)` | `.agg(count("*"), avg("duration"))` |
| Window functions | Same concept: `Window.partitionBy().orderBy()` |

**The mental model:** PySpark is SQL expressed as Python method chains. If you can write the SQL, you can write the PySpark.

---
## Hello World: PySpark in 5 Lines

Before diving into the full sections, here's the simplest possible PySpark program:

In [ ]:
# ============================================================
# Hello World: PySpark — read a CSV, filter, show results
# ============================================================

# WHY: This proves Spark works on your machine.
# If this runs, you're ready for Sections 18-22.

from pyspark.sql import SparkSession

# Create a Spark session (the entry point for all PySpark work)
# BEGINNER NOTE: SparkSession is like connecting to a database.
# You create it once, use it everywhere.
spark = SparkSession.builder.appName("HelloPySpark").getOrCreate()

# Create a simple DataFrame (like a SQL table in memory)
data = [
    ("C001", "8005551234", 120, "completed"),   # call_id, dnis, duration, disposition
    ("C002", "8005551234", 45, "transferred"),
    ("C003", "8005555678", 200, "completed"),
    ("C004", "8005555678", 15, "dropped"),
]
df = spark.createDataFrame(data, ["call_id", "dnis", "duration", "disposition"])

# Show all rows (like SELECT * FROM calls)
print("All calls:")
df.show()

# Filter (like WHERE duration > 60)
print("Calls longer than 60 seconds:")
df.filter(df.duration > 60).show()

# Aggregate (like SELECT dnis, COUNT(*), AVG(duration) GROUP BY dnis)
from pyspark.sql.functions import count, avg
print("Stats by campaign (DNIS = Dialed Number Identification Service):")
df.groupBy("dnis").agg(
    count("call_id").alias("total_calls"),
    avg("duration").alias("avg_duration")
).show()

# You should see:
# - 4 rows in the first table
# - 2 rows in the filtered table (C001 and C003)
# - 2 rows in the grouped table (one per DNIS)
#
# If this works, you're ready for the full PySpark sections.
# Stop Spark when done:
spark.stop()

---
## Checklist: Complete Before Starting M08

After working through Sections 18–22 and 25 in the M03 notebook, you should be able to:

- [ ] Create a SparkSession
- [ ] Read CSV, JSON, and Parquet files into DataFrames
- [ ] Filter, select, and transform columns
- [ ] Join two DataFrames (inner, left, anti)
- [ ] Group by and aggregate (count, sum, avg)
- [ ] Use window functions (row_number, rank, lag/lead)
- [ ] Write DataFrames to Parquet (with partitioning)
- [ ] Explain lazy evaluation (transformations vs actions)
- [ ] Read a query execution plan (`df.explain()`)
- [ ] Explain when to use broadcast joins vs shuffle joins

**If you can check all 10 boxes, proceed to M08.** If not, re-read the relevant section.

---
## What Comes Next: M08

In M08 (Cloud Data Pipeline — Build), you take everything you learned here and run it on cloud infrastructure:

```
M07 (this module, local):          M08 (cloud):

spark.read.json("data/calls.json")  -->  spark.read.json("gs://bucket/bronze/calls/")
df.write.parquet("/tmp/output")     -->  df.write.format("delta").save("gs://bucket/silver/calls")
Run on your laptop (8 cores)        -->  Run on Dataproc cluster (40 cores)
Query with df.show()                -->  Query with BigQuery (dashboard-ready)
```

**The PySpark code is identical.** Only the file paths and the cluster change.

**Open M08:**

[![Open M08 in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sunilmogadati/systems-in-production/blob/main/implementation/notebooks/Cloud_Pipeline_Reference.ipynb)